In [1]:
import pandas as pd
path = 'https://raw.githubusercontent.com/saivivekreddydevaram/EAMCET_ALLOTMENT_PREDICTOR/refs/heads/main/EMCET.csv'
data = pd.read_csv(path)
data.head()
data.columns
data_long = pd.melt(
    data,
    id_vars = ['INST_CODE','BRANCH','YEAR'],
    value_vars=['OCB', 'OCG', 'BCAB', 'BCAG', 'BCBB', 'BCBG', 'BCCB',
       'BCCG', 'BCDB', 'BCDG', 'BCEB', 'BCEG', 'SCB', 'SCG', 'STB', 'STG'],
    value_name='CUTOFF_RANK',
    var_name='CATEGORY'
)
data_long.dropna(subset=["CUTOFF_RANK"], inplace=True)
cap_level = 0.95
year_max = data_long.groupby('YEAR')['CUTOFF_RANK'].transform("max")
data_long["CAPPED"] = data_long["CUTOFF_RANK"] >= cap_level * year_max
data_long.head()
remove_branches = ["ARB","CEE","CS","CST","FDS","FPT","FSP","IPE","ITE","MNT","PHD","PHM"]
data_long = data_long[~data_long["BRANCH"].isin(remove_branches)].reset_index(drop=True)
data_long["GENDER"] = data_long["CATEGORY"].str[-1]
data_long["CASTE"]  = data_long["CATEGORY"].str[:-1]
last = data_long[["INST_CODE", "BRANCH", "CATEGORY", "YEAR", "CUTOFF_RANK", "CAPPED"]].copy()
last["YEAR"] = last["YEAR"] + 1
last = last.rename(columns={"CUTOFF_RANK": "LAST_CUTOFF", "CAPPED": "LAST_CAPPED"})
data_long = data_long.merge(last, on=["INST_CODE", "BRANCH", "CATEGORY", "YEAR"], how="left")
print(len(data_long))                      # should still be 110842
print(data_long["LAST_CUTOFF"].notna().sum())   # should be 87146
print(data_long.groupby("YEAR")["LAST_CUTOFF"].apply(lambda s: s.notna().sum()))
data_long.head()

110842
87146
YEAR
2018        0
2019    10810
2020    10552
2021    13176
2022    13024
2023    13088
2024    12936
2025    13560
Name: LAST_CUTOFF, dtype: int64


,INST_CODE,BRANCH,YEAR,CATEGORY,CUTOFF_RANK,CAPPED,GENDER,CASTE,LAST_CUTOFF,LAST_CAPPED
0,AARM,CSE,2019,OCB,25594.0,False,B,OC,21940.0,False
1,AARM,ECE,2019,OCB,51982.0,False,B,OC,33391.0,False
2,AARM,EEE,2019,OCB,99633.0,True,B,OC,38966.0,False
3,AARM,MEC,2019,OCB,29410.0,False,B,OC,41961.0,False
4,ACEG,CIV,2019,OCB,41557.0,False,B,OC,36728.0,False


In [2]:
import numpy as np

data_long["Y"] = np.log(data_long["CUTOFF_RANK"] / data_long["LAST_CUTOFF"])
data_long["ELIGIBLE"] = (data_long["CAPPED"] == False) & (data_long["LAST_CAPPED"] == False)
data_long.loc[~data_long["ELIGIBLE"], "Y"] = np.nan

In [3]:
from sklearn.metrics import mean_absolute_error

def score(true, pred):
    mae = mean_absolute_error(true, pred)                    # average miss, in rank numbers
    med = np.median(np.abs(pred - true) / true) * 100        # typical miss, in percent
    return round(mae), round(med, 1)

print("BASELINE: predict this year's cutoff = last year's cutoff")
for test_year in [2023, 2024, 2025]:
    train = data_long[(data_long["YEAR"] < test_year) & (data_long["Y"].notna())]
    test_all = data_long[(data_long["YEAR"] == test_year) & (data_long["LAST_CUTOFF"].notna())]
    test_open = test_all[test_all["CAPPED"] == False]

    a = score(test_all["CUTOFF_RANK"], test_all["LAST_CUTOFF"])
    b = score(test_open["CUTOFF_RANK"], test_open["LAST_CUTOFF"])
    print(f"{test_year}: train rows {len(train):6d} | test rows {len(test_all):6d} | "
          f"ALL rows MAE {a[0]:6d} median% {a[1]:5.1f} | "
          f"non-capped rows ({len(test_open)}) MAE {b[0]:6d} median% {b[1]:5.1f}")

BASELINE: predict this year's cutoff = last year's cutoff
2023: train rows  33123 | test rows  13088 | ALL rows MAE  31411 median%  24.8 | non-capped rows (8646) MAE  22673 median%  22.7
2024: train rows  41273 | test rows  12936 | ALL rows MAE  24555 median%  14.7 | non-capped rows (11338) MAE  23813 median%  16.5
2025: train rows  49310 | test rows  13560 | ALL rows MAE  25991 median%  24.0 | non-capped rows (11825) MAE  26241 median%  27.1


In [4]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

data_long["LOG_LAST"] = np.log(data_long["LAST_CUTOFF"])
features = ["BRANCH", "GENDER", "CASTE", "LOG_LAST"]

def run_ridge(test_year):
    train = data_long[(data_long["YEAR"] < test_year) & (data_long["Y"].notna())].copy()
    test  = data_long[(data_long["YEAR"] == test_year) & (data_long["LAST_CUTOFF"].notna())].copy()


    shift = train.groupby("YEAR")["Y"].median()
    train["Y_C"] = train["Y"] - train["YEAR"].map(shift)


    lo, hi = train["Y_C"].quantile([0.02, 0.98])
    train["Y_C"] = train["Y_C"].clip(lo, hi)

    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"),
                               ["BRANCH", "GENDER", "CASTE"])], remainder="passthrough")
    model = make_pipeline(prep, Ridge(alpha=1.0))
    model.fit(train[features], train["Y_C"])

    test["PRED"] = test["LAST_CUTOFF"] * np.exp(model.predict(test[features]))   # shift assumed 0
    return test

for ty in [2023, 2024, 2025]:
    t = run_ridge(ty)
    for name, sub in [("ALL rows", t), ("competitive (neither year capped)", t[t["ELIGIBLE"]])]:
        b = score(sub["CUTOFF_RANK"], sub["LAST_CUTOFF"])
        r = score(sub["CUTOFF_RANK"], sub["PRED"])
        print(f"{ty} {name:34s} baseline MAE {b[0]:6d} med% {b[1]:5.1f} | ridge MAE {r[0]:6d} med% {r[1]:5.1f}")

2023 ALL rows                           baseline MAE  31411 med%  24.8 | ridge MAE  36736 med%  30.4
2023 competitive (neither year capped)  baseline MAE  22640 med%  23.4 | ridge MAE  26239 med%  28.3
2024 ALL rows                           baseline MAE  24555 med%  14.7 | ridge MAE  26230 med%  20.2
2024 competitive (neither year capped)  baseline MAE  19951 med%  16.9 | ridge MAE  21167 med%  20.1
2025 ALL rows                           baseline MAE  25991 med%  24.0 | ridge MAE  19984 med%  18.9
2025 competitive (neither year capped)  baseline MAE  23764 med%  26.2 | ridge MAE  19575 med%  22.4


In [5]:

data_long["Y_ALL"] = np.log(data_long["CUTOFF_RANK"] / data_long["LAST_CUTOFF"])

multipliers = np.round(np.arange(0.5, 1.31, 0.1), 1)
test_years = [2023, 2024, 2025]
table = pd.DataFrame(index=multipliers)
table.index.name = "m"

for ty in test_years:
    past_Y = data_long[(data_long["YEAR"] < ty) & (data_long["Y"].notna())]["Y"]
    test = data_long[(data_long["YEAR"] == ty) & (data_long["LAST_CAPPED"] == False)]
    table[f"pred_{ty}"]   = [round((past_Y >= np.log(m)).mean(), 2) for m in multipliers]
    table[f"actual_{ty}"] = [round((np.log(m) <= test["Y_ALL"]).mean(), 2) for m in multipliers]

table["worst_actual"] = table[[f"actual_{ty}" for ty in test_years]].min(axis=1)
print(table.to_string())

     pred_2023  actual_2023  pred_2024  actual_2024  pred_2025  actual_2025  worst_actual
m                                                                                        
0.5       0.98         0.99       0.98         0.97       0.98         0.94          0.94
0.6       0.96         0.97       0.96         0.94       0.96         0.87          0.87
0.7       0.92         0.96       0.92         0.89       0.92         0.75          0.75
0.8       0.85         0.94       0.87         0.80       0.85         0.56          0.56
0.9       0.75         0.91       0.77         0.66       0.75         0.34          0.34
1.0       0.63         0.86       0.66         0.50       0.63         0.21          0.21
1.1       0.49         0.79       0.54         0.36       0.50         0.13          0.13
1.2       0.39         0.69       0.43         0.25       0.39         0.09          0.09
1.3       0.31         0.57       0.33         0.17       0.30         0.06          0.06


In [6]:
data.columns = data.columns.str.strip()
colleges = (data.sort_values("YEAR").drop_duplicates("INST_CODE", keep="last")
                [["INST_CODE", "INST_NAME", "PLACE", "DIST", "TYPE", "COED"]])
print(len(colleges))

277


In [7]:

SAFE_M, TARGET_M, REACH_M = 0.6, 0.9, 1.2
latest_year = data_long["YEAR"].max()
latest = data_long[data_long["YEAR"] == latest_year].merge(colleges, on="INST_CODE", how="left")
year_max_latest = latest["CUTOFF_RANK"].max()

def recommend(rank, caste, gender, branches=None, districts=None, shift=0.0, show_unlikely=False):
    """rank: student's rank | caste: OC, BCA, BCB, BCC, BCD, BCE, SC or ST | gender: 'B' or 'G'
       shift: expected change in cutoffs (0 = same as last year, -0.2 = tighter like 2025, +0.2 = looser)"""
    seats = latest[(latest["CASTE"] == caste) & (latest["GENDER"] == gender)].copy()
    if branches:  seats = seats[seats["BRANCH"].isin(branches)]
    if districts: seats = seats[seats["DIST"].isin(districts)]
    seats["EXPECTED_CUTOFF"] = (seats["CUTOFF_RANK"] * np.exp(shift)).round()
    seats["M"] = rank / seats["EXPECTED_CUTOFF"]

    def label(r):
        if r["CAPPED"]:
            return "Open (likely available)" if rank <= year_max_latest else "Unlikely"
        if r["M"] <= SAFE_M:   return "Safe"
        if r["M"] <= TARGET_M: return "Target"
        if r["M"] <= REACH_M:  return "Reach"
        return "Unlikely"

    seats["BUCKET"] = seats.apply(label, axis=1)
    if not show_unlikely: seats = seats[seats["BUCKET"] != "Unlikely"]
    order = {"Safe": 0, "Target": 1, "Reach": 2, "Open (likely available)": 3}
    seats["_o"] = seats["BUCKET"].map(order)
    seats = seats.sort_values(["_o", "CUTOFF_RANK"])
    return seats[["INST_NAME", "PLACE", "DIST", "BRANCH", "CUTOFF_RANK", "EXPECTED_CUTOFF", "BUCKET"]].reset_index(drop=True)

In [8]:
# CELL 3: try it
res = recommend(rank=14000, caste="BCB", gender="G", branches=["CSE","CSM"], districts=["HYD","MDL","RR"])
print(res["BUCKET"].value_counts().to_dict())
print(res.head(8).to_string())

{'Safe': 126, 'Target': 11, 'Open (likely available)': 8, 'Reach': 6}
                                          INST_NAME              PLACE DIST BRANCH  CUTOFF_RANK  EXPECTED_CUTOFF BUCKET
0  VIDYAJYOTHI INSTITUTE OF TECHNOLOGY (AUTONOMOUS)           MOINABAD   RR    CSE      23590.0          23590.0   Safe
1           BHOJREDDY ENGINEERING COLLEGE FOR WOMEN           SAIDABAD  HYD    CSM      24260.0          24260.0   Safe
2               KESHAV MEMORIAL ENGINEERING COLLEGE  KACHWANISING ARAM  MDL    CSM      24804.0          24804.0   Safe
3  VIDYAJYOTHI INSTITUTE OF TECHNOLOGY (AUTONOMOUS)           MOINABAD   RR    CSM      25063.0          25063.0   Safe
4        M L R INSTITUTE OF TECHNOLOGY (AUTONOMOUS)           DUNDIGAL  MDL    CSM      25980.0          25980.0   Safe
5               KESHAV MEMORIAL ENGINEERING COLLEGE  KACHWANISING ARAM  MDL    CSE      26484.0          26484.0   Safe
6           GOKARAJU LAILAVATHI ENGINEERING COLLEGE         BACHUPALLY  MDL    CSM      27

In [9]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

data_long["LOG_LAST"] = np.log(data_long["LAST_CUTOFF"])
features = ["BRANCH", "GENDER", "CASTE", "LOG_LAST"]
cat_cols = ["BRANCH", "GENDER", "CASTE"]
eligible_shift = data_long[data_long["ELIGIBLE"]].groupby("YEAR")["Y"].median()   # true yearly shift (reference only)

def make_model(name):
    if name == "Ridge (alpha=1)":
        return make_pipeline(ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough"), Ridge(alpha=1.0))
    if name == "Ridge (alpha=30000)":
        return make_pipeline(ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough"), Ridge(alpha=30000))
    if name == "Gradient boosting":
        return make_pipeline(ColumnTransformer([("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)], remainder="passthrough"),
                             HistGradientBoostingRegressor(max_depth=3, max_iter=100, learning_rate=0.05, min_samples_leaf=50, random_state=0))

def predict_with(name, test_year):
    train = data_long[(data_long["YEAR"] < test_year) & (data_long["Y"].notna())].copy()
    test  = data_long[(data_long["YEAR"] == test_year) & (data_long["LAST_CUTOFF"].notna())].copy()
    train["Y_C"] = train["Y"] - train["YEAR"].map(train.groupby("YEAR")["Y"].median())
    lo, hi = train["Y_C"].quantile([0.02, 0.98]); train["Y_C"] = train["Y_C"].clip(lo, hi)
    model = make_model(name).fit(train[features], train["Y_C"])
    return test, test["LAST_CUTOFF"] * np.exp(model.predict(test[features]))

results = []
for ty in [2023, 2024, 2025]:
    test = data_long[(data_long["YEAR"] == ty) & (data_long["LAST_CUTOFF"].notna())]
    entries = [("Baseline: reuse last year", test["LAST_CUTOFF"])]
    for name in ["Ridge (alpha=1)", "Ridge (alpha=30000)", "Gradient boosting"]:
        entries.append((name, predict_with(name, ty)[1]))
    entries.append(("Baseline x TRUE shift (not achievable)", test["LAST_CUTOFF"] * np.exp(eligible_shift[ty])))
    for name, pred in entries:
        mae, med = score(test["CUTOFF_RANK"], pred)
        within = round((np.abs(pred - test["CUTOFF_RANK"]) / test["CUTOFF_RANK"] <= 0.2).mean() * 100)
        results.append({"method": name, "year": ty, "MAE": mae, "median_%": med, "within_20%": within})

res = pd.DataFrame(results)
print(res.pivot(index="method", columns="year", values="median_%").to_string())
print(res.pivot(index="method", columns="year", values="MAE").to_string())
print(res.pivot(index="method", columns="year", values="within_20%").to_string())

year                                    2023  2024  2025
method                                                  
Baseline x TRUE shift (not achievable)  12.9  14.8  14.0
Baseline: reuse last year               24.8  14.7  24.0
Gradient boosting                       35.1  22.4  19.8
Ridge (alpha=1)                         30.4  20.2  18.9
Ridge (alpha=30000)                     25.7  16.9  21.5
year                                     2023   2024   2025
method                                                     
Baseline x TRUE shift (not achievable)  20409  24551  16942
Baseline: reuse last year               31411  24555  25991
Gradient boosting                       40370  27223  20131
Ridge (alpha=1)                         36736  26230  19984
Ridge (alpha=30000)                     32379  24822  22579
year                                    2023  2024  2025
method                                                  
Baseline x TRUE shift (not achievable)    63    60    63
Baseline: 

In [10]:
import joblib

latest_year = data_long["YEAR"].max()
latest = data_long[data_long["YEAR"] == latest_year].merge(colleges, on="INST_CODE", how="left")
latest = latest[["INST_CODE","INST_NAME","PLACE","DIST","TYPE","COED","BRANCH",
                 "CATEGORY","GENDER","CASTE","CUTOFF_RANK","CAPPED"]]

joblib.dump({
    "latest": latest,
    "safe_m": 0.6, "target_m": 0.9, "reach_m": 1.2,
    "year_max_latest": float(latest["CUTOFF_RANK"].max()),
    "latest_year": int(latest_year),
}, "recommender_data.pkl")

print(len(latest), latest_year)     # 14512 rows, 2025

14512 2025


In [12]:
%%writefile recommender.py
import joblib
import numpy as np
import pandas as pd

_art = joblib.load("recommender_data.pkl")
latest = _art["latest"]
SAFE_M, TARGET_M, REACH_M = _art["safe_m"], _art["target_m"], _art["reach_m"]
year_max_latest = _art["year_max_latest"]
latest_year = _art["latest_year"]


def recommend(rank, caste, gender, branches=None, districts=None, shift=0.0, show_unlikely=False):
    seats = latest[(latest["CASTE"] == caste) & (latest["GENDER"] == gender)].copy()
    if branches:
        seats = seats[seats["BRANCH"].isin(branches)]
    if districts:
        seats = seats[seats["DIST"].isin(districts)]
    seats["EXPECTED_CUTOFF"] = (seats["CUTOFF_RANK"] * np.exp(shift)).round()
    seats["M"] = rank / seats["EXPECTED_CUTOFF"]

    def label(r):
        if r["CAPPED"]:
            return "Open (likely available)" if rank <= year_max_latest else "Unlikely"
        if r["M"] <= SAFE_M:
            return "Safe"
        if r["M"] <= TARGET_M:
            return "Target"
        if r["M"] <= REACH_M:
            return "Reach"
        return "Unlikely"

    seats["BUCKET"] = seats.apply(label, axis=1)
    if not show_unlikely:
        seats = seats[seats["BUCKET"] != "Unlikely"]
    order = {"Safe": 0, "Target": 1, "Reach": 2, "Open (likely available)": 3}
    seats["_o"] = seats["BUCKET"].map(order)
    seats = seats.sort_values(["_o", "CUTOFF_RANK"])
    return seats[["INST_NAME", "PLACE", "DIST", "BRANCH", "CUTOFF_RANK", "EXPECTED_CUTOFF", "BUCKET"]].reset_index(drop=True)

Writing recommender.py


In [13]:
import recommender
res = recommender.recommend(rank=14000, caste="BCB", gender="G",
                            branches=["CSE","CSM"], districts=["HYD","MDL","RR"])
print(res["BUCKET"].value_counts().to_dict())

{'Safe': 126, 'Target': 11, 'Open (likely available)': 8, 'Reach': 6}


In [17]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/eamcet_project"
!cp recommender.py recommender_data.pkl "/content/drive/MyDrive/eamcet_project/"
!ls "/content/drive/MyDrive/eamcet_project"

Mounted at /content/drive
recommender_data.pkl  recommender.py


In [18]:
%%writefile app.py
import streamlit as st
import recommender as rec   # loads recommender_data.pkl automatically

st.set_page_config(page_title="TGEAPCET College Recommender", layout="wide")
st.title("TGEAPCET College Recommender")
st.caption(f"Based on {rec.latest_year} final-phase closing ranks. "
           "This is a guide, not a guarantee: next year's cutoffs can move a lot.")

latest = rec.latest

col1, col2, col3 = st.columns(3)
rank = col1.number_input("Your rank", min_value=1, max_value=300000, value=14000, step=100)
caste = col2.selectbox("Category", sorted(latest["CASTE"].unique()))
gender = col3.selectbox("Gender", ["B", "G"],
                        format_func=lambda g: "Boy" if g == "B" else "Girl")

branches = st.multiselect("Branches (leave empty for all)", sorted(latest["BRANCH"].unique()))
districts = st.multiselect("Districts (leave empty for all)", sorted(latest["DIST"].dropna().unique()))

shift = st.slider(
    "Expected change in cutoffs vs last year (log scale)",
    min_value=-0.30, max_value=0.30, value=0.0, step=0.05,
    help="0 = same as last year. Negative = cutoffs get smaller. "
         "Yearly changes have ranged from about -0.2 to +0.2 and cannot be predicted.")

show_unlikely = st.checkbox("Also show 'Unlikely' options", value=False)

if st.button("Find colleges"):
    res = rec.recommend(rank, caste, gender,
                        branches=branches or None,
                        districts=districts or None,
                        shift=shift, show_unlikely=show_unlikely)
    if res.empty:
        st.warning("No options found. Try more branches or districts.")
    else:
        counts = res["BUCKET"].value_counts()
        st.write("  |  ".join(f"**{b}**: {counts.get(b, 0)}"
                              for b in ["Safe", "Target", "Reach", "Open (likely available)"]))
        for bucket in ["Safe", "Target", "Reach", "Open (likely available)", "Unlikely"]:
            part = res[res["BUCKET"] == bucket]
            if len(part):
                with st.expander(f"{bucket} ({len(part)})", expanded=(bucket in ["Safe", "Target"])):
                    st.dataframe(part.drop(columns="BUCKET"), use_container_width=True)

st.markdown("---")
st.caption("Safe/Target/Reach come from a backtest on 2019-2025: students ranked at 0.6x of a seat's "
           "previous cutoff were admitted at least 87% of the time in the worst year. "
           "Ranks exclude special categories (PH, NCC, sports, CAP, minority). SC uses the SC_III "
           "series for 2025, a known limitation.")

Writing app.py


In [19]:
!cp app.py requirements.txt "/content/drive/MyDrive/eamcet_project/"
!ls "/content/drive/MyDrive/eamcet_project"

cp: cannot stat 'requirements.txt': No such file or directory
app.py	recommender_data.pkl  recommender.py


In [20]:
import recommender as rec
out = rec.recommend(14000, "BCB", "G", branches=["CSE","CSM"], districts=["HYD","MDL","RR"])
print(out["BUCKET"].value_counts())
out.head()

BUCKET
Safe                       126
Target                      11
Open (likely available)      8
Reach                        6
Name: count, dtype: int64


,INST_NAME,PLACE,DIST,BRANCH,CUTOFF_RANK,EXPECTED_CUTOFF,BUCKET
0,VIDYAJYOTHI INSTITUTE OF TECHNOLOGY (AUTONOMOUS),MOINABAD,RR,CSE,23590.0,23590.0,Safe
1,BHOJREDDY ENGINEERING COLLEGE FOR WOMEN,SAIDABAD,HYD,CSM,24260.0,24260.0,Safe
2,KESHAV MEMORIAL ENGINEERING COLLEGE,KACHWANISING ARAM,MDL,CSM,24804.0,24804.0,Safe
3,VIDYAJYOTHI INSTITUTE OF TECHNOLOGY (AUTONOMOUS),MOINABAD,RR,CSM,25063.0,25063.0,Safe
4,M L R INSTITUTE OF TECHNOLOGY (AUTONOMOUS),DUNDIGAL,MDL,CSM,25980.0,25980.0,Safe
